In [27]:
from utils.paths import MODELS_DIR,RAW_DIR

In [30]:
import pandas as pd
import numpy as np
import pickle

from preprocessing.DescriptorEngineer import AlloyDescriptorCalculator,ElementPropertyLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

In [10]:
df = pd.read_csv("selected_data.csv")


cols = ['r','del_r','del_EN','S','VEC']
X = df[cols]
y = df['resistivity']

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [16]:
def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

Train XGBoost

In [14]:
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

In [19]:
mae, rmse, r2 = evaluate(y_test, y_pred_xgb)

In [ ]:
xgb_model.save_model(MODELS_DIR/"xgb_selected_model.json")

Prediciton CuNiAl

In [37]:
comp_space = pd.read_csv(RAW_DIR/"CuNiAl_descriptors.csv")

x_space = comp_space[cols]
y_pred_space= xgb_model.predict(x_space)